In [5]:
# -----------------------------------------------------------
#  BOOT: cria banco em memória e injeta a versão 1 do esquema
# -----------------------------------------------------------
import sqlite3, json
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

# --- Versão do esquema -------------------------------------
cur.execute('PRAGMA user_version = 1')

cur.executescript("""
CREATE TABLE category (
  id   INTEGER PRIMARY KEY AUTOINCREMENT,
  name TEXT    NOT NULL
);
CREATE TABLE item (
  id          INTEGER PRIMARY KEY AUTOINCREMENT,
  name        TEXT    NOT NULL,
  category_id INTEGER NOT NULL REFERENCES category(id),
  price       REAL    NOT NULL,
  tags_json   TEXT,
  photo_path  TEXT
);
""")

# Seed de categorias e itens de inventário
cur.execute("INSERT INTO category(name) VALUES ('Bebidas'), ('Mercearia')")
cur.execute("""INSERT INTO item(name, category_id, price, tags_json, photo_path)
               VALUES (?,?,?,?,?)""",
            ('Café Torrado', 1, 19.90,
             json.dumps({"origem": "Minas", "tipo": "Arábica"}),
             'photos/cafe.jpg'))
cur.execute("""INSERT INTO item(name, category_id, price, tags_json, photo_path)
               VALUES (?,?,?,?,?)""",
            ('Biscoito Integral', 2, 7.50,
             json.dumps({"semGluten": False, "vegano": True}),
             'photos/biscoito.jpg'))
conn.commit()


In [6]:
# ---------------------------------------------
#  AUDITORIA: inspeção de esquema e amostras v1
# ---------------------------------------------
schema_v1  = pd.read_sql_query("PRAGMA table_info(item)", conn)
sample_v1  = pd.read_sql_query("SELECT id, name, price, tags_json FROM item", conn)

display(schema_v1)   # estrutura da tabela item (v1)
display(sample_v1)   # amostra de registros


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,name,TEXT,1,None,0
2,2,category_id,INTEGER,1,None,0
3,3,price,REAL,1,None,0
4,4,tags_json,TEXT,0,None,0
5,5,photo_path,TEXT,0,None,0


,id,name,price,tags_json
0,1,Café Torrado,19.9,"{""origem"": ""Minas"", ""tipo"": ""Ar\u00e1bica""}"
1,2,Biscoito Integral,7.5,"{""semGluten"": false, ""vegano"": true}"


In [7]:
# ------------------------------------------------------------
#  MIGRAÇÃO ATÔMICA: barcode (v2) + externalização de fotos (v3)
# ------------------------------------------------------------
cur.executescript("""
BEGIN;                                   -- garante atomicidade
/* v2 -------------------------------------------------------- */
ALTER TABLE item ADD COLUMN barcode TEXT;
PRAGMA user_version = 2;

/* v3 -------------------------------------------------------- */
CREATE TABLE IF NOT EXISTS item_photo (
  id         INTEGER PRIMARY KEY AUTOINCREMENT,
  item_id    INTEGER NOT NULL REFERENCES item(id),
  photo_path TEXT    NOT NULL
);
INSERT INTO item_photo(item_id, photo_path)
  SELECT id, photo_path FROM item;

/* Nota: SQLite não permite DROP COLUMN direto;        *
 * em produção usar técnica de tabela sombra + rename. */
PRAGMA user_version = 3;
COMMIT;
""")


In [8]:
# ----------------------------------------------------
#  AUDITORIA FINAL: conferência de esquema e dados v3
# ----------------------------------------------------
schema_v3 = pd.read_sql_query("PRAGMA table_info(item)", conn)
photos_v3 = pd.read_sql_query("SELECT * FROM item_photo", conn)

display(schema_v3)   # tabela item agora com 'barcode'
display(photos_v3)   # fotos migradas para item_photo


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,name,TEXT,1,None,0
2,2,category_id,INTEGER,1,None,0
3,3,price,REAL,1,None,0
4,4,tags_json,TEXT,0,None,0
5,5,photo_path,TEXT,0,None,0
6,6,barcode,TEXT,0,None,0


,id,item_id,photo_path
0,1,1,photos/cafe.jpg
1,2,2,photos/biscoito.jpg
